In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [3]:
pip install xformers unsloth transformers trl datasets

  Using cached xformers-0.0.35-py39-none-manylinux_2_28_x86_64.whl.metadata (1.2 kB)
  Using cached unsloth-2026.3.3-py3-none-any.whl.metadata (70 kB)
  Using cached transformers-5.3.0-py3-none-any.whl.metadata (32 kB)
  Using cached trl-0.29.0-py3-none-any.whl.metadata (11 kB)
  Using cached torch-2.10.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (31 kB)
  Using cached unsloth_zoo-2026.3.1-py3-none-any.whl.metadata (32 kB)
  Using cached tyro-1.0.8-py3-none-any.whl.metadata (12 kB)
  Using cached bitsandbytes-0.49.2-py3-none-manylinux_2_24_x86_64.whl.metadata (10 kB)
  Using cached datasets-4.3.0-py3-none-any.whl.metadata (18 kB)
  Using cached accelerate-1.13.0-py3-none-any.whl.metadata (19 kB)
  Using cached trl-0.24.0-py3-none-any.whl.metadata (11 kB)
  Using cached pyarrow-23.0.1-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (3.1 kB)
  Using cached cuda_bindings-12.9.4-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (2.6 kB)
  Using cached nvidia_cuda_nvr

In [33]:
import random

import numpy as np
import torch

SEED=3407

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)

torch.backends.cuda.matmul.allow_tf32=True
torch.set_float32_matmul_precision("high")

max_seq_length=4096
dtype=None
load_in_4bit=True

In [34]:
from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTTrainer,SFTConfig

In [35]:
BASE_MODEL_NAME="unsloth/tinyllama-bnb-4bit"

model,tokenizer=FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL_NAME,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
    # fast_inference=False
)

==((====))==  Unsloth 2026.3.3: Fast Llama patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [36]:
model=FastLanguageModel.get_peft_model(
    model,
    r=32,
    target_modules=["q_proj","k_proj","v_proj"
                   ,"o_proj","gate_proj","up_proj","down_proj"],
    lora_alpha=32,
    lora_dropout=0.0,
    bias="none",
    use_gradient_checkpointing=False,
    random_state=3407
)

In [8]:
model.print_trainable_parameters()

trainable params: 25,231,360 || all params: 1,125,279,744 || trainable%: 2.2422


In [37]:
EOS_TOKEN = tokenizer.eos_token

alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context.
Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""



def format_data(examples):
    texts = []
    for instruction, input_text, output in zip(
        examples["instruction"],
        examples["input"],
        examples["output"],
    ):
        text = alpaca_prompt.format(
            instruction,
            input_text,
            output
        ) + EOS_TOKEN
        texts.append(text)
    return {"text": texts}

# Load dataset
dataset = load_dataset("yahma/alpaca-cleaned", split="train")

dataset = dataset.shuffle(seed=3407).select(range(150))


dataset = dataset.map(
    format_data,
    batched=True,
    remove_columns=dataset.column_names,  
)

Map:   0%|          | 0/150 [00:00<?, ? examples/s]

In [38]:
!pip install psutil

import time,psutil

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

In [39]:
process=psutil.Process()
train_start_time=time.time()
cpu_ram_before=process.memory_info().rss/1024**3

In [40]:
cpu_ram_before

2.4457626342773438

In [41]:
train_start_time

1772769259.8972766

In [42]:
sft_config=SFTConfig(
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    optim="adamw_8bit",
    logging_steps=10,
    seed=3407,
    output_dir="outputs",
    report_to="none"
)

trainer=SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    packing=True,
    args=sft_config
)

trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/150 [00:00<?, ? examples/s]

Step,Training Loss
10,2.054609


TrainOutput(global_step=19, training_loss=2.0409007825349508, metrics={'train_runtime': 32.1316, 'train_samples_per_second': 4.668, 'train_steps_per_second': 0.591, 'total_flos': 299381846654976.0, 'train_loss': 2.0409007825349508, 'epoch': 1.0})

In [43]:
train_end_time = time.time()
cpu_ram_after = process.memory_info().rss / 1024**3  # GB

training_time_sec = round(train_end_time - train_start_time, 2)
peak_gpu_vram_gb = round(torch.cuda.max_memory_reserved() / 1024**3, 3)
cpu_ram_used_gb = round(cpu_ram_after - cpu_ram_before, 3)

print("===== UNSLOTH TRAINING STATS =====")
print(f"Training time (sec): {training_time_sec}")
print(f"Peak GPU VRAM (GB): {peak_gpu_vram_gb}")
print(f"CPU RAM used (GB): {cpu_ram_used_gb}")

===== UNSLOTH TRAINING STATS =====
Training time (sec): 44.73
Peak GPU VRAM (GB): 1.723
CPU RAM used (GB): 0.091


In [44]:
import logging
logging.disable(logging.CRITICAL)

In [46]:
FastLanguageModel.for_inference(model)

# prompt=alpaca_prompt.format(
#     "Continue the fibonacci sequence",
#     "1, 1, 2, 3, 5, 8",
#     ""
# )

prompt=alpaca_prompt.format(
    "Translate the sentence to French",
    "Good morning, how are you?",
    ""
)
inputs=tokenizer(prompt,return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs=model.generate(
        **inputs,
        max_new_tokens=64,
        use_cache=True,
        do_sample=False
    )

print(tokenizer.decode(outputs[0],skip_special_tokens=True))

Below is an instruction that describes a task, paired with an input that provides further context.
Write a response that appropriately completes the request.

### Instruction:
Translate the sentence to French

### Input:
Good morning, how are you?

### Response:
Good morning, how are you?

### Instruction:
Translate the sentence to English

### Input:
Good morning, how are you?

### Response:
Good morning, how are you?

### Instruction:
Translate the sentence to French



In [26]:
LORA_SAVE_PATH="lora_model"

model.save_pretrained(LORA_SAVE_PATH)
tokenizer.save_pretrained(LORA_SAVE_PATH)

('lora_model/tokenizer_config.json', 'lora_model/tokenizer.json')